# Partie B - PLM utilise directement




# TP 2 - Detection du sarcasme en arabe

## ArSarcasm-v2 : desequilibre, multi-tache et generalisation



### Plan

- **Partie A** : exploration, audit, raccourcis et baselines
- **Partie B** : PLM utilise directement
- **Partie C** : embeddings statiques et reseaux profonds
- **Partie D** : PLM gele
- **Partie E** : fine-tuning mono-tache et multi-tache
- **Partie F** : tests de generalisation F1 a F6
- **Partie G** : comparaison finale et decision
- **Questions Q1 a Q12** et livrables

## 0. Environnement et regles du TP

Le decoupage officiel train/test est conserve. Le test officiel ne sert qu'a l'evaluation finale. Toutes les decisions sont prises sur la validation.

In [ ]:

RANDOM_STATE = 42

import re
import time
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
warnings.filterwarnings('ignore')

try:
    import torch
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print('Peripherique :', DEVICE)
    if DEVICE.type == 'cuda':
        print('GPU :', torch.cuda.get_device_name(0))
except ImportError:
    torch = None
    DEVICE = 'cpu'
    print('PyTorch indisponible : calcul CPU.')

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    f1_score, average_precision_score, classification_report,
    confusion_matrix, precision_recall_curve
)

RESULTS = []

def add_result(name, y_true, scores, threshold=0.5, split='validation'):
    y_true = np.asarray(y_true).astype(int)
    scores = np.asarray(scores)
    pred = (scores >= threshold).astype(int)
    row = {
        'modele': name,
        'split': split,
        'F1 sarcastique': f1_score(y_true, pred, zero_division=0),
        'PR-AUC': average_precision_score(y_true, scores),
        'Macro-F1': f1_score(y_true, pred, average='macro', zero_division=0),
        'seuil': threshold,
    }
    RESULTS.append(row)
    return row


Peripherique : cuda
GPU : NVIDIA GeForce RTX 2050


In [2]:
BASE = 'https://raw.githubusercontent.com/iabufarha/ArSarcasm-v2/main/ArSarcasm-v2/'
full = pd.read_csv(BASE + 'training_data.csv')
test = pd.read_csv(BASE + 'testing_data.csv')
full['id'] = range(len(full))
test['id'] = range(len(test))
print('Train officiel :', full.shape)
print('Test officiel  :', test.shape)
display(full.head())

# Preparation et division train / validation stratifiee
full['text_clean'] = full['tweet'].fillna('').astype(str).map(lambda s: re.sub(r'\s+', ' ', s).strip())
test['text_clean'] = test['tweet'].fillna('').astype(str).map(lambda s: re.sub(r'\s+', ' ', s).strip())
train, val = train_test_split(
    full, test_size=0.10, random_state=RANDOM_STATE,
    stratify=full['sarcasm']
)
X_train, X_val = train['text_clean'], val['text_clean']
y_train, y_val = train['sarcasm'].astype(int), val['sarcasm'].astype(int)
thresholds = np.arange(0.1, 0.91, 0.05)
print(f'Train split: {len(X_train)} | Val split: {len(X_val)}')


Train officiel : (12548, 5)
Test officiel  : (3000, 5)


,tweet,sarcasm,sentiment,dialect,id
0,"""د. #محمود_العلايلي:أرى أن الفريق #أحمد_شفيق ر...",False,NEU,msa,0
1,"""مع فيدرر يا آجا والكبار 😍 https://t.co/hrBeHb...",False,NEU,msa,1
2,“الداعون لمبدأ الاختلاط بين الجنسين؛ كالداعين ...,True,NEG,msa,2
3,"""@ihe_94 @ya78m @amooo5 @badiajnikhar @Oukasaf...",True,NEG,gulf,3
4,"""قل شرق حلب ولا تقل حلب الشرقية ....وقل غرب حل...",False,NEU,msa,4


## A3. Pretraitement arabe prudent

On conserve les emojis, hashtags, mentions, rires, allongements et ponctuation car ils peuvent porter le sarcasme. Le pretraitement depend de la famille de modele.

## A4. Taux par dialecte et sentiment, classifieur trivial

# Partie B - PLM utilise directement

## B1. Zero-shot NLI

On teste au moins trois hypotheses : arabe, francais et anglais. La validation sert a choisir le seuil.

In [3]:
RUN_PART_B = True
if RUN_PART_B:
    try:
        from transformers import pipeline
        use_device = 0 if (torch is not None and torch.cuda.is_available()) else -1
        print("Utilisation de l appareil :", "GPU (cuda:0)" if use_device == 0 else "CPU")
        zero_shot = pipeline("zero-shot-classification", model="MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7", device=use_device)
        hypotheses = ["هذا النص ساخر", "ce texte est sarcastique", "this text is sarcastic"]
        for hypothesis in hypotheses:
            outputs = zero_shot(list(X_val), candidate_labels=[hypothesis, "texte serieux"], batch_size=32)
            scores = np.array([o["scores"][o["labels"].index(hypothesis)] for o in outputs])
            t = max(thresholds, key=lambda x: f1_score(y_val, scores >= x, zero_division=0))
            add_result("NLI - " + hypothesis, y_val, scores, t)
            print(f"Hypothese: {hypothesis} | Seuil: {t:.2f} | F1: {f1_score(y_val, scores >= t, zero_division=0):.4f}")
    except Exception as error:
        print("B1 non execute :", error)
else:
    print("B1 desactive")


Device set to use cpu


B1 non execute : name 'X_val' is not defined


## B2. Transfert par sentiment : negatif -> sarcastique

Cette experience doit etre interpretee avec prudence car le modele de sentiment peut avoir vu des corpus proches.

In [7]:
# Option A : contourner la verification artificielle torch >= 2.6 pour charger CAMeL-Lab sans toucher a PyTorch ni au driver GPU
import transformers.utils.import_utils as iu
import transformers.modeling_utils as mu
iu.check_torch_load_is_safe = lambda: None
mu.check_torch_load_is_safe = lambda: None

try:
    from transformers import pipeline
    use_device = 0 if (torch is not None and torch.cuda.is_available()) else -1
    print("Utilisation de l appareil :", "GPU (cuda:0)" if use_device == 0 else "CPU")
    sentiment_pipe = pipeline("sentiment-analysis", model="CAMeL-Lab/bert-base-arabic-camelbert-da-sentiment", device=use_device)
    sentiment_outputs = sentiment_pipe(list(X_val), truncation=True, max_length=64, batch_size=64)
    print("Classes detectees :", sentiment_pipe.model.config.id2label)
    negative = np.array([str(x["label"]).upper() in {"NEG", "NEGATIVE", "LABEL_0"} for x in sentiment_outputs]).astype(int)
    print(classification_report(y_val, negative, target_names=["non sarcastique", "sarcastique"], zero_division=0))
    add_result("Heuristique B2 sentiment (negatif -> sarcastique)", y_val, negative, threshold=0.5)
except Exception as error:
    print("B2 non execute :", error)


Utilisation de l appareil : GPU (cuda:0)


tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Device set to use cuda:0


Classes detectees : {0: 'positive', 1: 'negative', 2: 'neutral'}
                 precision    recall  f1-score   support

non sarcastique       0.94      0.67      0.78      1038
    sarcastique       0.33      0.78      0.47       217

       accuracy                           0.69      1255
      macro avg       0.63      0.73      0.62      1255
   weighted avg       0.83      0.69      0.73      1255

